# Gemini 4 Carbon on the paper's test sets: population β MSE (see `gemini-4-carbon-beta-test.md`)

## Paths and imports

In [1]:
import asyncio
import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from google import genai
from google.genai import types
from IPython.display import display
from tenacity import retry, retry_if_exception, stop_after_attempt, wait_random_exponential
from tqdm.auto import tqdm

# Walk up to the repository's `code/` root, starting from this notebook's folder (VS Code sets __vsc_ipynb_file__),
# otherwise from the working directory; works again after the chdir below
if "__vsc_ipynb_file__" in globals():
    search_start = Path(__vsc_ipynb_file__).resolve().parent
else:
    search_start = Path.cwd()
CODE_DIR = None
for candidate in (search_start, *search_start.parents):
    if all((candidate / d).is_dir() for d in ("kev", "data", "models")):
        CODE_DIR = candidate
        break
if CODE_DIR is None:
    raise RuntimeError(f"No code/ root containing kev/, data/ and models/ at or above {search_start}")
GEMINI_DIR = CODE_DIR / "proprietary-llms" / "gemini"
# `beta_ols` comes from the sim-respondents paper's copy (code/papers/sim-respondents/src)
BETA_OLS_SRC_DIR = CODE_DIR / "papers" / "sim-respondents" / "src"
# Run from `code/` so data loaders and shared disk caches resolve as in the training notebooks
os.chdir(CODE_DIR)
for import_dir in (CODE_DIR, CODE_DIR / "kev", BETA_OLS_SRC_DIR, GEMINI_DIR):
    if str(import_dir) not in sys.path:
        sys.path.insert(0, str(import_dir))

import beta_ols
from beta_ols.evaluation import build_population_eval, compare_population_betas, flat_probs_from_predictions
from data.grummon2025 import GRUMMON2025_PERSONA_NAMES
from data.persona import PERSONA_NAMES
from gemini_api import is_transient_api_error

## Design shared by all runs

In [2]:
# Calls go to Gemini Enterprise Agent Platform (formerly Vertex AI), global endpoint, not Google AI Studio.
# Signs in with Application Default Credentials; the Google Cloud project is read from GOOGLE_CLOUD_PROJECT.
PLATFORM = "Gemini Enterprise Agent Platform"
PLATFORM_LOCATION = "global"
# List price in USD per 1M tokens on the global endpoint (standard pay-as-you-go, not launch promotions); thinking
# tokens are billed at the output price. Check Google's pricing page before running.
PRICES_PER_MILLION_TOKENS = {
    "gemini-4-carbon": {"input": 4.00, "output": 20.00},
    "gemini-4-neon": {"input": 3.00, "output": 15.00},
    "gemini-4-hydrogen": {"input": 1.50, "output": 7.50},
}
# Flights (picks KEV's best.pt), Vaccines and Food labels (held-out test sets)
DATASETS = ["kriner2026", "covid", "grummon2025"]
SYSTEM_PROMPT_HEAD = "You are a survey respondent with the profile below. Answer the question as this person would."
# Readable persona labels from the data package; the country column is left out
PERSONA_LABELS = {
    "kriner2026": PERSONA_NAMES,
    "covid": PERSONA_NAMES,
    "grummon2025": GRUMMON2025_PERSONA_NAMES,
}
OUTPUT_ROOT = GEMINI_DIR / "data_beta_full"


# One folder per (model, thinking level): answers from different settings never mix
def run_dir_for(model, thinking_level):
    if thinking_level is None:
        thinking_label = "default"
    else:
        thinking_label = thinking_level
    return OUTPUT_ROOT / f"{model}_{thinking_label}", thinking_label

## Prompts: persona, question and options read from each dataset's long parquet, one prompt per task

In [3]:
DATASETS = ['covid']

In [4]:
def build_prompt(row, persona_cols, persona_labels):
    persona_lines = []
    for col in persona_cols:
        if pd.notna(row[col]):
            persona_lines.append(f"{persona_labels[col]}: {row[col]}")
    system_prompt = SYSTEM_PROMPT_HEAD + "\n\n" + "\n".join(persona_lines)
    option_lines = []
    for option_code, option_text, option_description in zip(
        row["option_codes"], row["option_texts"], row["option_descriptions"], strict=True
    ):
        if pd.isna(option_description):
            option_lines.append(f"{option_code}. {option_text}")
        else:
            option_lines.append(f"{option_code}. {option_text}: {option_description}")
    user_prompt = row["question"] + "\n\n" + "\n".join(option_lines)
    return system_prompt, user_prompt


# load_choice_df reads <dataset>_US_long.parquet and keeps the rows of the study's choice task (one row per task)
choice_dfs = {}
records_by_dataset = {}
prompt_items_by_dataset = {}
for dataset_name in DATASETS:
    dataset_module = beta_ols.DATASET_MODULES[dataset_name]
    choice_df = dataset_module.load_choice_df()
    rows = choice_df.to_dict("records")
    # A respondent can see the same task twice (same CASEID and qid): the occurrence number tells the two apart
    occurrences = {}
    # Respondents are numbered 1, 2, 3, ... in the order they first appear in the parquet
    respondent_numbers = {}
    prompt_items = []
    for row in rows:
        caseid_qid = (str(row["CASEID"]), str(row["qid"]))
        occurrence = occurrences.get(caseid_qid, 0)
        occurrences[caseid_qid] = occurrence + 1
        if caseid_qid[0] not in respondent_numbers:
            respondent_numbers[caseid_qid[0]] = len(respondent_numbers) + 1
        system_prompt, user_prompt = build_prompt(row, dataset_module.PERSONA_COLS, PERSONA_LABELS[dataset_name])
        option_codes = []
        for option_code in row["option_codes"]:
            option_codes.append(str(option_code))
        prompt_items.append(
            {
                "key": f"{caseid_qid[0]}|{caseid_qid[1]}|{occurrence}",
                "respondent_number": respondent_numbers[caseid_qid[0]],
                "system_prompt": system_prompt,
                "user_prompt": user_prompt,
                "options": option_codes,
            }
        )
    choice_dfs[dataset_name] = choice_df
    records_by_dataset[dataset_name] = rows
    prompt_items_by_dataset[dataset_name] = prompt_items
    print(f"{dataset_name}: {len(respondent_numbers):,} respondents, {len(prompt_items):,} tasks")

for dataset_name in DATASETS:
    first_item = prompt_items_by_dataset[dataset_name][0]
    print(f"\n===== {dataset_name}")
    print("--- SYSTEM ---\n" + first_item["system_prompt"])
    print("--- USER ---\n" + first_item["user_prompt"])

covid: 1,971 respondents, 9,855 tasks

===== covid
--- SYSTEM ---
You are a survey respondent with the profile below. Answer the question as this person would.

Region: Northeast
Sex: Female
Age: 30-49
Education: Associate's degree
Religion: Agnostic
Political ideology: Somewhat liberal
Political party: Independent
Income: $60,000 - $79,999
Race: White
--- USER ---
As you may know, scientists around the world are working to develop a vaccine for Covid-19. Please consider the hypothetical vaccines described below:

If you had to choose, would you choose to get Vaccine A or Vaccine B, or would you choose not to be vaccinated?

1. Vaccine A: Efficacy (protection against severe symptoms): 90%; Protection duration: 5 years; Risk of severe side effects (hospitalization or death): 1 in 10,000; Risk of mild side effects (flu-like symptoms): 1 in 30; Government authorization: The vaccine has been approved and licensed by the US Food and Drug Administration.; Vaccine origin: United Kingdom; Reco

In [5]:
print(first_item["system_prompt"])

You are a survey respondent with the profile below. Answer the question as this person would.

Region: Northeast
Sex: Female
Age: 30-49
Education: Associate's degree
Religion: Agnostic
Political ideology: Somewhat liberal
Political party: Independent
Income: $60,000 - $79,999
Race: White


In [6]:
print(first_item["user_prompt"])

As you may know, scientists around the world are working to develop a vaccine for Covid-19. Please consider the hypothetical vaccines described below:

If you had to choose, would you choose to get Vaccine A or Vaccine B, or would you choose not to be vaccinated?

1. Vaccine A: Efficacy (protection against severe symptoms): 90%; Protection duration: 5 years; Risk of severe side effects (hospitalization or death): 1 in 10,000; Risk of mild side effects (flu-like symptoms): 1 in 30; Government authorization: The vaccine has been approved and licensed by the US Food and Drug Administration.; Vaccine origin: United Kingdom; Recommended by: President Donald Trump
2. Vaccine B: Efficacy (protection against severe symptoms): 70%; Protection duration: 1 year; Risk of severe side effects (hospitalization or death): 1 in 1,000,000; Risk of mild side effects (flu-like symptoms): 1 in 10; Government authorization: The vaccine has been approved and licensed by the US Food and Drug Administration.; 

## Gemini call: the answer is restricted to the task's option codes

In [7]:
@retry(
    retry=retry_if_exception(is_transient_api_error),
    wait=wait_random_exponential(min=1, max=60),
    stop=stop_after_attempt(8),
    reraise=True,
)
async def ask_gemini(client, item, model, thinking_level, thinking_label, price, attempt_counter):
    # attempt_counter counts every try of this task, including the retries tenacity makes
    attempt_counter["attempts"] += 1
    answer_schema = {
        "type": "object",
        "properties": {"choice": {"type": "string", "enum": item["options"]}},
        "required": ["choice"],
    }
    if thinking_level is None:
        thinking_config = types.ThinkingConfig(include_thoughts=True)
    else:
        thinking_config = types.ThinkingConfig(include_thoughts=True, thinking_level=thinking_level)
    config = types.GenerateContentConfig(
        system_instruction=item["system_prompt"],
        response_mime_type="application/json",
        response_json_schema=answer_schema,
        thinking_config=thinking_config,
    )
    # call_seconds: time of this one request, from sending it to receiving the full answer (perf_counter is monotonic)
    started_at_utc = datetime.now(timezone.utc).isoformat()
    call_start = time.perf_counter()
    response = await client.aio.models.generate_content(model=model, contents=item["user_prompt"], config=config)
    call_seconds = time.perf_counter() - call_start
    if response.text is None or not response.candidates or response.candidates[0].content is None:
        raise ValueError(f"{item['key']}: incomplete Gemini response {response!r}")
    if response.usage_metadata is None:
        raise ValueError(f"{item['key']}: Gemini response has no token counts")

    # response.text holds only the answer parts; thought summaries are separate parts
    choice = json.loads(response.text)["choice"]
    if choice not in item["options"]:
        raise ValueError(f"{item['key']}: answer {choice!r} is not one of {item['options']}")
    reasoning_parts = []
    for part in response.candidates[0].content.parts or []:
        if part.thought and part.text:
            reasoning_parts.append(part.text)

    usage = response.usage_metadata
    prompt_tokens = int(usage.prompt_token_count)
    output_tokens = int(usage.candidates_token_count or 0)
    thought_tokens = int(usage.thoughts_token_count or 0)
    input_cost = prompt_tokens * price["input"] / 1_000_000
    output_cost = (output_tokens + thought_tokens) * price["output"] / 1_000_000
    return {
        "key": item["key"],
        "model": model,
        "thinking_level": thinking_label,
        "choice": choice,
        "raw_response": response.text,
        "reasoning_text": "".join(reasoning_parts),
        "prompt_tokens": prompt_tokens,
        "output_tokens": output_tokens,
        "thought_tokens": thought_tokens,
        "total_tokens": int(usage.total_token_count),
        "input_price_per_million": price["input"],
        "output_price_per_million": price["output"],
        "cost_usd": input_cost + output_cost,
        "started_at_utc": started_at_utc,
        "call_seconds": call_seconds,
    }

## Query helpers: answers are appended to `<dataset>.jsonl`; a task already in the file is never asked again

In [8]:
def load_answers(jsonl_path, model, thinking_label):
    answers = {}
    if not jsonl_path.exists():
        return answers
    with jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            answer = json.loads(line)
            if answer["model"] != model or answer["thinking_level"] != thinking_label:
                raise ValueError(f"{jsonl_path}: answer from {answer['model']} / {answer['thinking_level']}")
            answers[answer["key"]] = answer
    return answers


async def query_dataset(client, dataset_name, caseids, model, thinking_level, max_concurrency):
    # Asks every task of the respondents in caseids (the parquet's CASEID values) that is not answered yet.
    # thinking_level: None = the model's default; otherwise "MINIMAL", "LOW", "MEDIUM" or "HIGH".
    price = PRICES_PER_MILLION_TOKENS[model]
    run_dir, thinking_label = run_dir_for(model, thinking_level)
    run_dir.mkdir(parents=True, exist_ok=True)
    run_design = {
        "model": model,
        "thinking_level": thinking_label,
        "platform": PLATFORM,
        "platform_location": PLATFORM_LOCATION,
        "price_per_million_tokens": price,
        "system_prompt_head": SYSTEM_PROMPT_HEAD,
        "prompt_source": "<dataset>_US_long.parquet rows from load_choice_df; persona labels from data.persona / data.grummon2025",
        "answer_format": "JSON schema, enum of the task's option codes",
        "samples_per_task": 1,
    }
    (run_dir / "run.json").write_text(json.dumps(run_design, indent=2))

    jsonl_path = run_dir / f"{dataset_name}.jsonl"
    answers = load_answers(jsonl_path, model, thinking_label)
    all_items = prompt_items_by_dataset[dataset_name]
    caseid_set = set(caseids)
    dataset_caseids = set()
    for item in all_items:
        dataset_caseids.add(item["key"].split("|")[0])
    unknown_caseids = caseid_set - dataset_caseids
    if len(unknown_caseids) > 0:
        raise ValueError(f"{dataset_name}: CASEIDs not in the dataset: {sorted(unknown_caseids)[:5]}")
    target_items = []
    for item in all_items:
        if item["key"].split("|")[0] in caseid_set:
            target_items.append(item)
    pending_items = [item for item in target_items if item["key"] not in answers]
    print(f"{jsonl_path}: {len(answers):,} of {len(all_items):,} tasks answered")
    print(f"{len(caseid_set):,} respondents: {len(target_items):,} tasks, {len(pending_items):,} to ask")

    failures = []
    if len(pending_items) > 0:
        semaphore = asyncio.Semaphore(max_concurrency)
        progress = tqdm(total=len(pending_items), desc=dataset_name, unit="task")
        with jsonl_path.open("a", encoding="utf-8") as f_out:

            # task_seconds: from the moment the task gets a concurrency slot (so queueing behind other tasks is not
            # counted) until its answer arrives, including failed tries and the waits between retries
            async def ask_and_save(item):
                async with semaphore:
                    attempt_counter = {"attempts": 0}
                    task_start = time.perf_counter()
                    answer = await ask_gemini(client, item, model, thinking_level, thinking_label, price, attempt_counter)
                    answer["task_seconds"] = time.perf_counter() - task_start
                    answer["attempts"] = attempt_counter["attempts"]
                    answer["max_concurrency"] = max_concurrency
                answers[item["key"]] = answer
                f_out.write(json.dumps(answer, ensure_ascii=False) + "\n")
                f_out.flush()
                progress.update(1)

            # A failed task does not stop the others: every finished answer is written before the file closes
            run_started_at_utc = datetime.now(timezone.utc).isoformat()
            run_start = time.perf_counter()
            try:
                results = await asyncio.gather(*[ask_and_save(item) for item in pending_items], return_exceptions=True)
            finally:
                progress.close()
            run_seconds = time.perf_counter() - run_start
        for item, result in zip(pending_items, results, strict=True):
            if isinstance(result, BaseException):
                failures.append((item["key"], repr(result)))

        # run_seconds: wall-clock time of this whole query, what a researcher waits at this concurrency
        run_log = {
            "started_at_utc": run_started_at_utc,
            "run_seconds": run_seconds,
            "tasks_asked": len(pending_items),
            "tasks_answered": len(pending_items) - len(failures),
            "tasks_failed": len(failures),
            "max_concurrency": max_concurrency,
        }
        with (run_dir / f"{dataset_name}_query_runs.jsonl").open("a", encoding="utf-8") as f_log:
            f_log.write(json.dumps(run_log) + "\n")
        print(f"{dataset_name}: asked {len(pending_items):,} tasks in {run_seconds:,.1f} s at max_concurrency={max_concurrency}")

    total_cost = 0.0
    for answer in answers.values():
        total_cost += answer["cost_usd"]
    print(f"{dataset_name}: {len(answers):,} of {len(all_items):,} tasks answered, cost so far ${total_cost:,.2f}")
    if len(failures) > 0:
        for key, error in failures[:5]:
            print(f"  failed {key}: {error}")
        raise RuntimeError(f"{dataset_name}: {len(failures):,} tasks failed (not saved); rerun the cell to ask them again")


client = genai.Client(enterprise=True, project=os.environ["GOOGLE_CLOUD_PROJECT"], location=PLATFORM_LOCATION)

## Query Vaccines (covid)

In [9]:
# The 200 respondents of the earlier Gemini 4 Argon runs (data_sample1000/): seed 42 draws 200 of the sorted respondent
# ids without replacement, as in gemini-3.5-flash-lite.ipynb Step 3; the parquet's `respondent` is the same id
ARGON_SAMPLE_SEED = 42
ARGON_SAMPLE_SIZE = 200
covid_choice_df = choice_dfs["covid"]
all_respondent_ids = np.sort(covid_choice_df["respondent"].unique())
rng = np.random.default_rng(ARGON_SAMPLE_SEED)
argon_respondent_ids = np.sort(rng.choice(all_respondent_ids, size=ARGON_SAMPLE_SIZE, replace=False))
argon_rows = covid_choice_df[covid_choice_df["respondent"].isin(argon_respondent_ids)]
ARGON_SAMPLE_CASEIDS = sorted(argon_rows["CASEID"].unique())
print(f"{len(ARGON_SAMPLE_CASEIDS)} respondents, {len(argon_rows):,} tasks; first ids: {argon_respondent_ids[:10].tolist()}")

200 respondents, 1,000 tasks; first ids: [14, 45, 59, 67, 71, 81, 112, 116, 124, 141]


In [10]:
ARGON_SAMPLE_CASEIDS[:2]

['covid_1000', 'covid_1002']

In [11]:
await query_dataset(
    client, 
    "covid", 
    caseids=ARGON_SAMPLE_CASEIDS[:1],
    model="gemini-4-carbon", 
    thinking_level='HIGH', 
    max_concurrency=1,
)



/home/dw/github/decision-models-finetuning/code/proprietary-llms/gemini/data_beta_full/gemini-4-carbon_HIGH/covid.jsonl: 1 of 9,855 tasks answered
1 respondents: 5 tasks, 4 to ask


covid:   0%|          | 0/4 [00:00<?, ?task/s]

covid: asked 4 tasks in 195.4 s at max_concurrency=1
covid: 5 of 9,855 tasks answered, cost so far $0.49


# Estimate cost

In [13]:
# Average cost and time per answer in every run folder so far, scaled to one respondent, to the 200 Argon respondents
# (Vaccines only) and to the full dataset. Time estimate: mean call_seconds x tasks, i.e. the summed time of the calls.
# Only answers saved with timings are used for time.
estimate_rows = []
for run_dir in sorted(OUTPUT_ROOT.iterdir()):
    for dataset_name in DATASETS:
        jsonl_path = run_dir / f"{dataset_name}.jsonl"
        if not jsonl_path.exists():
            continue
        answers_df = pd.read_json(jsonl_path, lines=True)
        if len(answers_df) == 0:
            print(f"{jsonl_path}: no answers yet, left out")
            continue
        all_items = prompt_items_by_dataset[dataset_name]
        n_tasks = len(all_items)
        n_respondents = max(item["respondent_number"] for item in all_items)
        cost_per_task = answers_df["cost_usd"].mean()
        if "call_seconds" in answers_df.columns:
            call_seconds = answers_df["call_seconds"].dropna()
        else:
            call_seconds = pd.Series(dtype=float)
        if "task_seconds" in answers_df.columns:
            task_seconds = answers_df["task_seconds"].dropna()
        else:
            task_seconds = pd.Series(dtype=float)
        seconds_per_task = call_seconds.mean()
        if dataset_name == "covid":
            argon_caseid_set = set(ARGON_SAMPLE_CASEIDS)
            argon_sample_tasks = 0
            for item in all_items:
                if item["key"].split("|")[0] in argon_caseid_set:
                    argon_sample_tasks += 1
        else:
            argon_sample_tasks = np.nan
        estimate_rows.append(
            {
                "run": run_dir.name,
                "dataset": dataset_name,
                "answered_tasks": len(answers_df),
                "mean_prompt_tokens": answers_df["prompt_tokens"].mean(),
                "mean_output_tokens": answers_df["output_tokens"].mean(),
                "mean_thought_tokens": answers_df["thought_tokens"].mean(),
                "cost_so_far_usd": answers_df["cost_usd"].sum(),
                "cost_per_task_usd": cost_per_task,
                "cost_per_respondent_usd": cost_per_task * n_tasks / n_respondents,
                "argon_sample_tasks": argon_sample_tasks,
                "argon_sample_cost_usd": cost_per_task * argon_sample_tasks,
                "full_dataset_tasks": n_tasks,
                "full_dataset_cost_usd": cost_per_task * n_tasks,
                "timed_tasks": len(call_seconds),
                "mean_call_seconds": seconds_per_task,
                "median_call_seconds": call_seconds.median(),
                "p90_call_seconds": call_seconds.quantile(0.9),
                "max_call_seconds": call_seconds.max(),
                "mean_task_seconds": task_seconds.mean(),
                "argon_sample_hours": seconds_per_task * argon_sample_tasks / 3600,
                "full_dataset_hours": seconds_per_task * n_tasks / 3600,
            }
        )
estimate_df = pd.DataFrame(estimate_rows)
display(estimate_df)

,run,dataset,answered_tasks,mean_prompt_tokens,mean_output_tokens,mean_thought_tokens,cost_so_far_usd,cost_per_task_usd,cost_per_respondent_usd,argon_sample_tasks,...,full_dataset_tasks,full_dataset_cost_usd,timed_tasks,mean_call_seconds,median_call_seconds,p90_call_seconds,max_call_seconds,mean_task_seconds,argon_sample_hours,full_dataset_hours
0,gemini-4-carbon_HIGH,covid,5,596.4,22.4,4799.0,0.494068,0.098814,0.494068,1000,...,9855,973.808028,4,48.840445,45.719895,72.398705,76.614589,48.843377,13.56679,133.700717
1,gemini-4-carbon_HIGH_resp1,covid,4,596.5,22.5,4572.5,0.377144,0.094286,0.471430,1000,...,9855,929.188530,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,gemini-4-carbon_LOW_resp1,covid,5,598.0,22.0,0.0,0.014160,0.002832,0.014160,1000,...,9855,27.909360,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,gemini-4-carbon_MEDIUM_resp1,covid,5,598.0,22.0,442.8,0.058440,0.011688,0.058440,1000,...,9855,115.185240,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Scoring helper: population β from one-hot answers, same OLS operators as KEV; needs every task answered

In [10]:
def score_dataset(dataset_name, model, thinking_level):
    run_dir, thinking_label = run_dir_for(model, thinking_level)
    answers = load_answers(run_dir / f"{dataset_name}.jsonl", model, thinking_label)
    all_items = prompt_items_by_dataset[dataset_name]
    missing = [item for item in all_items if item["key"] not in answers]
    if len(missing) > 0:
        print(f"{dataset_name}: {len(missing):,} of {len(all_items):,} tasks not answered yet, not scored")
        return None

    dataset_module = beta_ols.DATASET_MODULES[dataset_name]
    choice_df = choice_dfs[dataset_name]
    rows = dataset_module.regression_rows(choice_df)
    specs = dataset_module.regression_specs(rows)
    eval_ops = build_population_eval(dataset_module, choice_df, specs, records_by_dataset[dataset_name])

    predictions = []
    for item in all_items:
        choice = answers[item["key"]]["choice"]
        one_hot = np.zeros(len(item["options"]))
        one_hot[item["options"].index(choice)] = 1.0
        predictions.append(one_hot)
    probs_flat = flat_probs_from_predictions(eval_ops, predictions)
    metrics, beta_df = compare_population_betas(eval_ops, probs_flat)

    answers_df = pd.DataFrame(list(answers.values()))
    metrics["dataset"] = dataset_name
    metrics["model"] = model
    metrics["thinking_level"] = thinking_label
    metrics["tasks"] = len(predictions)
    metrics["prompt_tokens"] = int(answers_df["prompt_tokens"].sum())
    metrics["output_tokens"] = int(answers_df["output_tokens"].sum())
    metrics["thought_tokens"] = int(answers_df["thought_tokens"].sum())
    metrics["cost_usd"] = float(answers_df["cost_usd"].sum())
    metrics_df = pd.DataFrame([metrics]).set_index("dataset")
    metrics_df.to_csv(run_dir / f"{dataset_name}_metrics.csv")
    beta_df.to_csv(run_dir / f"{dataset_name}_betas.csv")
    display(metrics_df)
    display(beta_df)
    return metrics_df

## Score Flights (kriner2026)

In [ ]:
kriner_metrics = score_dataset("kriner2026", model="gemini-4-carbon", thinking_level=None)

## Score Vaccines (covid)

In [11]:
covid_metrics = score_dataset("covid", model="gemini-4-carbon", thinking_level=None)

covid: 9,855 of 9,855 tasks not answered yet, not scored


## Score Food labels (grummon2025)

In [ ]:
grummon_metrics = score_dataset("grummon2025", model="gemini-4-carbon", thinking_level=None)